# Fase 2. Modelos de demanda

Este notebook revisa los resultados de `make train`. La lógica de entrenamiento, ajuste y registro está en `mibici/modeling`; aquí solo se consultan MLflow y el modelo registrado.

In [ ]:
import mlflow
import pandas as pd

from mibici.config import (
    MLFLOW_MODEL_EXPERIMENT,
    MLFLOW_TRACKING_URI,
    PANEL_FILE,
    REGISTERED_MODEL_NAME,
    TEST_START,
)
from mibici.features import TARGET
from mibici.modeling.evaluation import metrics_by_group, regression_metrics, temporal_split
from mibici.modeling.predict import FORECAST_COLUMN, DemandForecaster

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

## Partición temporal

In [ ]:
panel = pd.read_parquet(PANEL_FILE)
train, test = temporal_split(panel, TEST_START)
pd.DataFrame(
    {
        'filas': [len(train), len(test)],
        'desde': [train['hour_ts'].min(), test['hour_ts'].min()],
        'hasta': [train['hour_ts'].max(), test['hour_ts'].max()],
        'pct_horas_sin_viajes': [(train[TARGET] == 0).mean() * 100, (test[TARGET] == 0).mean() * 100],
    },
    index=['entrenamiento', 'prueba'],
).round(1)

## Comparación de modelos en MLflow

Un run por modelo y por línea base, con sus métricas en el periodo de prueba.

In [ ]:
runs = mlflow.search_runs(
    experiment_names=[MLFLOW_MODEL_EXPERIMENT], order_by=['attributes.start_time DESC']
)
runs = runs[runs['tags.tipo'].isin(['modelo', 'linea_base'])]
cols = ['tags.modelo', 'tags.tipo', 'metrics.cv_rmse', 'metrics.test_MAE', 'metrics.test_RMSE',
        'metrics.test_WAPE_pct', 'metrics.test_sesgo']
runs = runs.drop_duplicates('tags.modelo')  # la corrida más reciente de cada modelo
runs[cols].sort_values('metrics.test_MAE').round(3)

## Ajuste de hiperparámetros

Cada combinación probada es un run anidado. Se muestran las mejores por modelo.

In [ ]:
trials = mlflow.search_runs(
    experiment_names=[MLFLOW_MODEL_EXPERIMENT],
    filter_string="tags.tipo = 'combinacion'",
    order_by=['metrics.cv_rmse ASC'],
)
param_cols = [c for c in trials.columns if c.startswith('params.')]
trials.groupby('tags.modelo').head(3)[['tags.modelo', 'metrics.cv_rmse', 'metrics.cv_mae', *param_cols]]

## Modelo registrado

Versiones en el Model Registry y la que tiene el alias `champion`.

In [ ]:
client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
pd.DataFrame(
    [
        {
            'version': int(v.version),
            'modelo': v.tags.get('modelo'),
            'cv_rmse': v.tags.get('cv_rmse'),
            'test_MAE': v.tags.get('test_MAE'),
            'hiperparametros': v.tags.get('hiperparametros'),
        }
        for v in versions
    ]
).sort_values('version')

In [ ]:
forecaster = DemandForecaster()
forecast = forecaster.predict(test)
pd.Series(regression_metrics(forecast[TARGET], forecast[FORECAST_COLUMN])).round(3).to_frame('champion')

## Error por grupo de estación y por hora

In [ ]:
metrics_by_group(forecast[TARGET], forecast[FORECAST_COLUMN], test['station_cluster']).round(3)

In [ ]:
metrics_by_group(forecast[TARGET], forecast[FORECAST_COLUMN], test['hour'], 'hora').round(3)